# ODE Solutions to SPK Files

This tutorial shows how to convert ODE solution segments into files that can be used with SPICE-based analysis and visualization workflows.

The main function is `ode_sol_to_spk`. It writes a BSP/SPK file from one or more ODE solution segments using native SPICE SPK writing routines, together with optional maneuver and metadata sidecar files. For long station-keeping simulations, `append_ode_sol_to_spk!` appends one ODE solution at a time and `finalize_spk` writes the sidecar files at the end.

The SPK files contain type-13 segments, written directly through SPICE. No external SPK-writing executable is required.

## Running this notebook

The next cell activates the environment shipped with the examples (`examples/Project.toml`), which uses the local copy of `HighFidelityEphemerisModel.jl`. Skip it if you would rather use your own environment with `HighFidelityEphemerisModel`, `OrdinaryDiffEq`, and `SPICE` installed.

SPICE kernels are loaded from the directory given by the `SPICE` environment variable, which should contain the `lsk/`, `spk/`, `pck/`, and `fk/` subfolders of the [NAIF generic kernels](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/). Edit `spice_dir` below if your kernels live elsewhere.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## Setup

We first load the SPICE kernels and define the dynamics. Here, we consider a spacecraft in a 3000 km radius orbit around the Moon, perturbed by the Earth and the Sun.

In [ ]:
using LinearAlgebra
using OrdinaryDiffEq
using SPICE
using HighFidelityEphemerisModel

# load SPICE kernels
spice_dir = ENV["SPICE"]
furnsh(joinpath(spice_dir, "lsk", "naif0012.tls"))
furnsh(joinpath(spice_dir, "spk", "de440.bsp"))
furnsh(joinpath(spice_dir, "pck", "gm_de440.tpc"))

naif_ids = ["301", "399", "10"]        # Moon-centered, with Earth & Sun as third bodies
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 3000.0
et0 = str2et("2026-01-05T00:00:00")
parameters = SpiceParameters(et0, DU, GMs, naif_ids, "J2000", "NONE")

## Propagating coast arcs

We build a trajectory made of four 12-hour coast arcs. Impulsive maneuvers are applied at the start of the second and fourth arcs, so the velocity is discontinuous at these two boundaries. The second and third arcs are joined continuously (i.e. there is no maneuver between them).

In [ ]:
inc = deg2rad(60.0)
x0 = [1.0, 0.0, 0.0, 0.0, cos(inc), sin(inc)]   # initial state, in canonical units
arc_duration = 12 * 3600 / parameters.TU        # duration of each arc, in canonical time

# impulsive Δv applied at the start of each arc, in km/s
dv_kmps = [
    [0.0, 0.0,   0.0],
    [0.0, 0.005, 0.0],
    [0.0, 0.0,   0.0],
    [0.0, 0.0,  -0.005],
]

sols = []
for (k, dv) in enumerate(dv_kmps)
    # each arc starts where the previous one ended
    x_start = k == 1 ? copy(x0) : copy(sols[end].u[end])
    t_start = k == 1 ? 0.0 : sols[end].t[end]

    x_start[4:6] += dv / parameters.VU          # apply impulse
    prob = ODEProblem(eom_Nbody!, x_start, (t_start, t_start + arc_duration), parameters)
    push!(sols, solve(prob, Vern8(), reltol=1e-12, abstol=1e-12))
end
length(sols)

## Expected ODE solution format

The SPK writer expects each coast arc to behave like a standard `OrdinaryDiffEq.jl` solution:

```julia
sol.t       # nondimensional time grid
sol(t_nd)   # interpolated nondimensional state at time t_nd
```

Each state returned by `sol(t_nd)` should contain at least

```julia
[x, y, z, vx, vy, vz]
```

in nondimensional units. For example, for the first arc:

In [ ]:
t_nd = (sols[1].t[1] + sols[1].t[end]) / 2
sols[1](t_nd)

The parameter object must provide the canonical scaling values

In [ ]:
(
    TU = parameters.TU,   # seconds per nondimensional time unit
    DU = parameters.DU,   # kilometers per nondimensional distance unit
    VU = parameters.VU,   # kilometers/second per nondimensional velocity unit
)

Internally, states are converted before writing the SPK:

```julia
r_km   = x_nd[1:3] .* parameters.DU
v_kmps = x_nd[4:6] .* parameters.VU
et     = et0 + t_nd * parameters.TU
```

## Basic SPK generation

A minimal call looks like this (the generated files are written to a temporary directory; change `outdir` to keep them):

In [ ]:
outdir = mktempdir()
spice_id = -123456

result = ode_sol_to_spk(
    sols,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
)

Here:

- `sols` is a vector of ODE solution segments,
- `parameters` provides the scaling units and, as `parameters.et0`, the reference epoch of the nondimensional time (it can also be passed explicitly, as in `ode_sol_to_spk(sols, et0, parameters; ...)`),
- `spice_id` is the NAIF ID (or a name known to SPICE) assigned to the generated trajectory.

Everything else has a default:

- `center_id`, the center of motion, defaults to the central body of the dynamics, `parameters.naif_ids[1]` (here, the Moon),
- `ref_frame_name`, the frame of the states written to the SPK, defaults to the integration frame `parameters.naif_frame` (here, `J2000`),
- `dt_sec = 1800.0` is the sampling interval used for each SPK segment,
- `segment_gap_sec = 1e-7` trims the right endpoint of non-final segments to avoid overlapping SPK coverage at impulsive boundaries,
- `write_metadata = true` writes a metadata JSON file next to the BSP.

Sidecar files are named after the BSP: `trajectory_metadata.json`, `trajectory_maneuvers.txt` and `trajectory_trajectory_jump_maneuvers.txt` here. The returned `result` is a `NamedTuple` containing generated paths and summaries:

In [ ]:
for key in (:output_spk, :maneuver_txt, :trajectory_maneuver_txt, :metadata_json,
            :segment_count, :epoch_ranges)
    println(rpad(key, 24), "=> ", getproperty(result, key))
end

## BSP/SPK file

The main output is the BSP file:

In [ ]:
result.output_spk

By default, `ode_sol_to_spk` writes one SPK segment per ODE solution segment. This is useful for impulsive trajectories because it avoids interpolating across discontinuous velocity jumps.

If several ODE solution segments should be grouped into one SPK segment, pass explicit coast windows. Each tuple in `coast_windows` gives the first and last ODE solution segment included in that SPK segment. Only group segments that are continuous; in our trajectory, the second and third arcs can be merged:

In [ ]:
result_windows = ode_sol_to_spk(
    sols,
    parameters;
    output_spk = joinpath(outdir, "trajectory_windows.bsp"),
    spice_id = spice_id,
    coast_windows = [(1, 1), (2, 3), (4, 4)],
    write_metadata = false,
)
result_windows.segment_count

## Incremental append workflow

For station-keeping simulations, `append_ode_sol_to_spk!` appends one propagated ODE arc at a time to a BSP. This avoids storing every ODE solution in memory until the end of the run.

In the loop below, each `sol_recurse` stands in for the ODE solution of one recursion interval, which would typically be obtained by solving an OCP, applying the (possibly corrupted) first maneuver, and propagating the truth dynamics.

In [ ]:
output_spk = joinpath(outdir, "sk_seed_001_native_true.bsp")

for k in 1:length(sols)
    sol_recurse = sols[k]

    append_ode_sol_to_spk!(
        sol_recurse,
        parameters;
        output_spk = output_spk,
        spice_id = -200001,
        append = k > 1,
    )
end

With `append = false`, the first call creates the BSP (replacing a file left by a previous run); later calls append new SPK type-13 segments to it. Segment `k` is named `HFEM_SPK_SEGMENT_00k` (change the prefix with `segment_id`). Where consecutive arcs share a boundary epoch, SPICE uses the segment appended last, i.e. the post-maneuver arc.

`append_ode_sol_to_spk!` writes no sidecar files. Once the loop is over, `finalize_spk` reads the segments of the BSP and writes a metadata file describing all of them, as well as the maneuver file when `ocp_control` and `ocp_control_times` (see below) are given:

In [ ]:
result_sk = finalize_spk(output_spk, parameters)
result_sk.segment_count, result_sk.metadata_json

## Maneuver files

There are two maneuver-file concepts.

### Primary executed/OCP maneuver file

For optimal-control or station-keeping workflows, the main maneuver product can be written from a control history. Here, we build the control history from the impulses applied to our trajectory: column `k` of `ocp_control` holds the nondimensional Δv applied at the nondimensional time `ocp_control_times[k]`.

In [ ]:
idx_maneuvers = findall(dv -> norm(dv) > 0, dv_kmps)        # arcs starting with an impulse
ocp_control = hcat([dv_kmps[k] / parameters.VU for k in idx_maneuvers]...)
ocp_control_times = [sols[k].t[1] for k in idx_maneuvers]

result = ode_sol_to_spk(
    sols,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    ocp_control = ocp_control,
    ocp_control_times = ocp_control_times,
)
print(read(result.maneuver_txt, String))

Rows 1:3 of `ocp_control` are the vector control components. If row 4 is present, it is treated as the scalar control/slack variable used by the OCP formulation. The scalar convention is summarized in metadata.

For station-keeping truth products, pass the actually applied/corrupted control history as the `ocp_control` input. In that case, the maneuver file represents executed maneuvers, not the originally planned control.

### Trajectory-jump diagnostic maneuver file

If `write_maneuvers = true`, the helper can also write a diagnostic maneuver file reconstructed from velocity jumps between adjacent ODE solution segments:

In [ ]:
result = ode_sol_to_spk(
    sols,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    write_maneuvers = true,
)
print(read(result.trajectory_maneuver_txt, String))

The diagnostic file is separate from the executed/OCP control file. It contains:

```text
# ETSECONDS,DVX_mps,DVY_mps,DVZ_mps
```

The reconstructed velocity jumps match the 5 m/s impulses applied at the start of the second and fourth arcs, while the jump at the continuous boundary between the second and third arcs is at the level of numerical noise.

Trajectory-jump diagnostics need all the ODE solutions, so they are only available from `ode_sol_to_spk`, not from `finalize_spk`.

## Metadata JSON

If `write_metadata = true` (the default), a metadata sidecar file is written:

In [ ]:
result.metadata_json

The metadata file records information such as:

- generated product paths,
- SPICE object ID and name, center ID and name,
- reference frame, SPK type and polynomial degree,
- segment coverage windows (with segment IDs and number of samples),
- sampling interval and segment gap,
- nondimensional scaling values,
- force-model information,
- maneuver summaries (and, with `merge_maneuvers_into_metadata = true`, the individual maneuvers).

The force-model information is read from `parameters`: bodies and GMs, integration frame, spherical harmonics (central body, degree, body-fixed frame), SRP and drag. Entries that `parameters` does not store, such as the name of the equations of motion, can be added with `force_model_metadata`, which is merged into the inferred entries:

In [ ]:
result = ode_sol_to_spk(
    sols,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    force_model_metadata = Dict("dynamics" => "eom_Nbody!"),
)
print(read(result.metadata_json, String))

For instance, the SRP coefficients `Cr`, `A/m` and `P0` are folded into `parameters.k_srp_cannonball`; to record them, pass

```julia
force_model_metadata = Dict(
    "dynamics" => "eom_NbodySH!",
    "SRP" => Dict("Cr" => 1.15, "A_over_m" => 0.016, "P0" => 4.56e-6),
)
```

## Label files

A label file is a lightweight sidecar file used by downstream visualization or analysis scripts to associate the generated BSP with an object name, SPICE ID, center, frame, and display settings.

The information needed to create such a label file is recorded in the `"spk"` entry of the metadata file (object and center IDs and names, reference frame). The exact label format depends on the downstream tool, so a label-writing step is kept separate from `ode_sol_to_spk`.

## Validating the generated BSP

A generated BSP can be validated by furnishing it with SPICE, querying states from the BSP, and comparing those states against the original ODE solution:

In [ ]:
furnsh(result.output_spk)

try
    for (k, sol) in enumerate(sols)
        # query an epoch near the middle of the k-th arc, between two SPK samples
        # (default dt_sec = 1800 s), so that the SPK interpolation is exercised
        et_query = et0 + (sol.t[1] + sol.t[end]) / 2 * parameters.TU + 600.0
        state_spk, _ = spkezr(
            string(spice_id),
            et_query,
            "J2000",
            "NONE",
            "MOON",
        )

        t_nd = (et_query - et0) / parameters.TU
        x_ref = sol(t_nd)

        state_ref = vcat(
            x_ref[1:3] .* parameters.DU,
            x_ref[4:6] .* parameters.VU,
        )

        @assert isapprox(state_spk[1:3], state_ref[1:3]; atol = 1e-2)   # km
        @assert isapprox(state_spk[4:6], state_ref[4:6]; atol = 1e-5)   # km/s
        println("arc $k: position error = $(norm(state_spk[1:3] - state_ref[1:3])) km, ",
                "velocity error = $(norm(state_spk[4:6] - state_ref[4:6])) km/s")
    end
finally
    unload(result.output_spk)
end

At the sampled epochs the SPK reproduces the ODE solution to round-off; between samples, the error is set by the type-13 Hermite interpolation of states sampled every `dt_sec` (here, errors of a few meters and a few mm/s with the default `dt_sec = 1800.0` s). Decrease `dt_sec` if a tighter agreement is needed.

This is the same validation strategy used in the automated end-to-end tests. The tests generate small BSPs with native SPICE, furnish them, query states from the BSP, compare them against the original ODE solutions, unload the BSPs, and check that temporary files are cleaned up.

Finally, we remove the temporary output directory:

In [ ]:
rm(outdir; recursive = true)